### Classify Articles into Field

The plan is to use CESifo WP Series Meta Data as training data and then classify the published papers into fields based on their title and abstract.
The output should be one field category per paper, not multiple.

My initial training data has a lot of information on the WPs. I assign them into ten distinct categories first, one per paper, because some have several and the original data also has way more distinct categories.
I cluster them into 10 categories which I then label by inspecting them.

Then I train a classifier on the labeled working papers. I have N working papers each with a single cluster label and a title + abstract. Train a standard text classifier on this. The input is title + abstract, the output is one of the 10 categories.

Then I run the trained classifier on my articles (e.g. their title and abstract). Each is assigned one of the ten categories.






In [ ]:
# Install dependencies (run once)
# !pip install sentence-transformers scikit-learn pandas numpy matplotlib seaborn umap-learn os

In [1]:
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
N_CLUSTERS = 10

# Set Working Directory
os.chdir(r"D:\Projektfolder1 (Miethe, Grosenick)\zz_AmelieMisc\thankyou")



---
## STEP 1: Load and prepare WP Data (Training Data)

In [2]:
df_wp = pd.read_excel(
    r"data/cesifo_wp/QS-CESifo-Working-Paper-Dateien-alle-bis-12407.xlsx",
    header=0  # top row as column names
)

print(df_wp.shape)
print(df_wp.columns.tolist())
print(df_wp.head())

(12392, 9)
['Artikelnummer', 'Title', 'Autoren (alt.)', 'Serien', 'Downloaddatei https://www.ifo.de/DocDL/…', 'Erscheinungsjahr', 'JEL Codes', 'Schlüsselwörter', 'WP Kategorien']
    Artikelnummer                                              Title  \
0  12012026012407  How Parenting Styles Shape Children’s Lifetime...   
1  12012026012405  When Music is Made by AI: Effects on Preferenc...   
2  12012026012404                Technology and Economic Development   
3  12012026012403  Task-Specific Technical Change and Comparative...   
4  12012026012402  Heterogeneous Individual Valuations of a Mild ...   

                                      Autoren (alt.)  \
0  Dohmen, Thomas / Golsteyn, Bart / Grönqvist, H...   
1  Friedrichsen, Jana / Schwarz, Julia / Clement,...   
2   Acemoglu, Daron / Akcigit, Ufuk / Johnson, Simon   
3                   Althoff, Lukas / Reichardt, Hugo   
4                                  Brueckner, Jan K.   

                           Serien Downloaddatei htt

In [3]:
# rename columns
df_wp.rename(columns={
    'Title': 'title',
    'Abstract': 'abstract',
    'Schlüsselwörter': 'keywords',
    'JEL Codes': 'jel_codes',
    'Autoren (alt.)': 'authors',
    'Artikelnummer': 'wp_id',
    'Serien': 'wp_series',
    'Downloaddatei https://www.ifo.de/DocDL/…': 'link',
    'Erscheinungsjahr': 'year',
    'WP Kategorien': 'categories'}, inplace=True)

print(df_wp.columns.tolist())


['wp_id', 'title', 'authors', 'wp_series', 'link', 'year', 'jel_codes', 'keywords', 'categories']


In [ ]:
# abstracts are only in PDFs, not in Excel with Meta-Data
# I extract the text from the PDFs and add it to a column called 'abstract'

from PyPDF2 import PdfReader

# path to the folder containing the PDFs
pdf_folder = r"data/cesifo_wp/pdfs"

def extract_abstracts_from_pdfs(pdf_folder):
    """
    Extract abstracts from all PDFs in a folder.
    Returns a DataFrame with filename and abstract columns.
    """
    results = []
    
    # Get all PDF files in folder
    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
    
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_folder, pdf_file)
        
        try:
            reader = PdfReader(pdf_path)
            
                
            # Extract text from pages 1-3 (indices 0-2)
            if len(reader.pages) > 2:
                text = ""
                for page_idx in range(3):  # pages 0, 1, 2
                    page = reader.pages[page_idx]
                    text += page.extract_text()
                
                text = re.sub(r'\s+', ' ', text) # remove line breaks and extra white space

                # Search for abstract and JEL (allows for white space in-between, e.g. caused by line breaks)
                start_idx = re.search(r'a\s*b\s*s\s*t\s*r\s*a\s*c\s*t', text, re.IGNORECASE).start()
                end_idx = re.search(r'j\s*e\s*l', text, re.IGNORECASE).start()
                
                if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
                    abstract = text[start_idx + len('abstract'):end_idx].strip()
                else:
                    abstract = "Abstract not found"
            else:
                abstract = "PDF has fewer than 3 pages"
        
        except Exception as e:
            abstract = f"Error: {str(e)}"
        
        results.append({'name': pdf_file, 'abstract': abstract})
    
    return pd.DataFrame(results)

# Apply the function
df_abstracts = extract_abstracts_from_pdfs(pdf_folder)
print(df_abstracts)

In [ ]:
error_abstracts = df_abstracts[
    (df_abstracts['abstract'].str.contains("Error", na=False)) & 
    (df_abstracts['abstract'].str.len() < 150)
]
print(error_abstracts)  # Show all unique error messages

# about 600 cannot be done; drop them

In [ ]:
# drop if error in abstract extraction
df_abstracts_clean = df_abstracts[
    (~df_abstracts['abstract'].str.contains("Error", na=False)) & 
    (df_abstracts['abstract'].str.len() >= 150)
]

In [ ]:
# cheeck for which ones I could not find abstracts
print(df_abstracts_clean[df_abstracts_clean['abstract'] == "Abstract not found"])

# also drop
df_abstracts_clean = df_abstracts_clean[df_abstracts_clean['abstract'] != "Abstract not found"]	
print(df_abstracts_clean.shape)

In [4]:
# save the data set with abstracts
# df_abstracts_clean.to_csv(r"data/cesifo_wp/abstracts.csv", index=False)	

# load the data set with abstracts
df_abstracts_clean = pd.read_csv(r"data/cesifo_wp/abstracts.csv")
print(df_abstracts_clean.columns)
print(df_abstracts_clean.head())

Index(['name', 'abstract'], dtype='object')
                    name                                           abstract
0     cesifo1_wp1000.pdf  This paper studies the joint effect of fr acti...
1    cesifo1_wp10000.pdf  We estimate the impact of the Covid-induced sh...
2  cesifo1_wp10000_0.pdf  We estimate the impact of the Covid-induced sh...
3    cesifo1_wp10001.pdf  This paper examines how perceived importance o...
4    cesifo1_wp10003.pdf  We consider a principal -agent relationship wi...


In [5]:
print(df_wp.columns.tolist())
print(df_wp['link'].head)

['wp_id', 'title', 'authors', 'wp_series', 'link', 'year', 'jel_codes', 'keywords', 'categories']
<bound method NDFrame.head of 0          cesifo1_wp12407.pdf (1.33 MB)
1          cesifo1_wp12405.pdf (1.28 MB)
2          cesifo1_wp12404.pdf (7.71 MB)
3          cesifo1_wp12403.pdf (2.66 MB)
4        cesifo1_wp12402.pdf (361.78 KB)
                      ...               
12387            ces_wp5.pdf (602.13 KB)
12388            ces_wp4.pdf (626.31 KB)
12389              ces_wp3.pdf (5.33 MB)
12390              ces_wp2.pdf (6.15 MB)
12391            ces_wp1.pdf (932.71 KB)
Name: link, Length: 12392, dtype: object>


In [6]:
# merge abstracts to meta data using wp_number as unique identifier

df_wp['wp_number'] = df_wp['link'].str.extract(r'(\d+)\.pdf') # get working paper number as unique identifier
print(df_wp[['wp_id', 'wp_number']].head())

df_abstracts_clean['wp_number'] = df_abstracts_clean['name'].str.extract(r'wp(\d+)')
print(df_abstracts_clean[['name', 'wp_number']].head())

# Convert to numeric for proper sorting/merging
df_wp['wp_number'] = pd.to_numeric(df_wp['wp_number'], errors='coerce')
df_abstracts_clean['wp_number'] = pd.to_numeric(df_abstracts_clean['wp_number'], errors='coerce')

# Merge
df_wp_abstracts = pd.merge(df_wp, df_abstracts_clean[['name', 'abstract', 'wp_number']], 
                           on='wp_number', how='left')

# Check results
print(df_wp_abstracts[df_wp_abstracts['abstract'].notna()].head())
print(df_wp_abstracts[df_wp_abstracts['abstract'].notna()].shape)

# Count rows with and without abstracts
with_abstract = df_wp_abstracts['abstract'].notna().sum()
without_abstract = df_wp_abstracts['abstract'].isna().sum()

print(f"With abstract: {with_abstract}")
print(f"Without abstract: {without_abstract}")
print(f"Total: {len(df_wp_abstracts)}")

# Or as percentage
print(f"Percentage with abstract: {with_abstract / len(df_wp_abstracts) * 100:.1f}%")

# count how many have duplicate based on wp_number (different versions)
duplicates = df_wp_abstracts['wp_number'].duplicated().sum()
print(f"Duplicate wp_numbers: {duplicates}")

# if duplicate wp number, only keep most recent
df_wp_abstracts['wp_id'] = df_wp_abstracts['wp_id'].astype(str)
df_wp_abstracts['publication_date'] = pd.to_datetime(
    df_wp_abstracts['wp_id'].astype(str).str[:8], 
    format='%d%m%Y'
) # get publication date as first 8 letters from wp_id

# if you are a doplicate keep only the most recent one
print(df_wp_abstracts.shape)
df_wp_abstracts = df_wp_abstracts.sort_values('publication_date').drop_duplicates('wp_number', keep='last')
print(df_wp_abstracts.shape)



            wp_id wp_number
0  12012026012407     12407
1  12012026012405     12405
2  12012026012404     12404
3  12012026012403     12403
4  12012026012402     12402
                    name wp_number
0     cesifo1_wp1000.pdf      1000
1    cesifo1_wp10000.pdf     10000
2  cesifo1_wp10000_0.pdf     10000
3    cesifo1_wp10001.pdf     10001
4    cesifo1_wp10003.pdf     10003
              wp_id                                              title  \
132  12012025012272  Policies to Decarbonize Industry While Address...   
133  12012025012271                 How Global Are Local Value Chains?   
135  12012025012269  Evaluating Online Exercise Solution Videos Ver...   
136  12012025012268  Informal Institutions and Global Trade: Real E...   
138  12012025012266  The Causal Effects of Trump's Reelection on Bu...   

                                               authors  \
132  Parry, Ian / Black, Simon / Minnett, Danielle ...   
133  Borin, Alessandro / Conteduca, Francesco Paolo...   
135

In [7]:
print(f"Before cleaning: {len(df_wp_abstracts):,} working papers")

# Drop rows missing title or abstract — these can't be embedded (should all be due to missing abstract)
df_wp_abstracts = df_wp_abstracts.dropna(subset=['title', 'abstract'])
df_wp_abstracts = df_wp_abstracts.reset_index(drop=True)

# Fill other columns with empty string if missing
for col in ['jel_codes', 'keywords', 'categories']:
    df_wp_abstracts[col] = df_wp_abstracts[col].fillna('')

print(f"After cleaning: {len(df_wp_abstracts):,} working papers")

Before cleaning: 12,152 working papers
After cleaning: 11,090 working papers


In [8]:
# Sentence transformer expects a single text string as input

# Build the text input for embedding.
# I concatenate title + abstract + JEL codes + keywords so the
# clustering picks up on all available signals.

def combine_text(row):
    parts = [
        row['title'],
        row['abstract'],
        row['jel_codes'],
        row['keywords'],
    ]
    return ' '.join(p for p in parts if p.strip())

df_wp_abstracts['text_for_embedding'] = df_wp_abstracts.apply(combine_text, axis=1)


# Preview
print(df_wp_abstracts['text_for_embedding'].iloc[0][:300])

Property Rights, Finance, and Entrepreneurship Is investment constrained more by insecure property rights or by limited external finance? For five transition economies in Eastern Europe and the former Soviet Union we find that weak property rights limit the reinvestment of profits in startup manufac


---
## STEP 2: Embed & Cluster into 10 Categories

We use **SPECTER** — a language model trained specifically on scientific paper title+abstract pairs. It produces 768-dimensional embeddings that place papers with similar content close together in vector space.

Then K-Means assigns each paper to exactly one of 10 clusters.

In [13]:
# Load SPECTER model (downloads ~420MB on first run)
# Alternative if slow: 'sentence-transformers/all-MiniLM-L6-v2' (faster, slightly less accurate)
print("Loading SPECTER model...")
model = SentenceTransformer('allenai-specter')
print("Model loaded.")

Loading SPECTER model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

: 

In [ ]:
# Generate embeddings for all working papers
# This may take a few minutes depending on corpus size
print(f"Embedding {len(df_wp_abstracts):,} working papers...")

wp_embeddings = model.encode(
    df_wp_abstracts['text_for_embedding'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embedding matrix shape: {wp_embeddings.shape}")

# Save embeddings so you don't have to recompute
np.save('wp_embeddings.npy', wp_embeddings)
print("Embeddings saved to wp_embeddings.npy")

In [ ]:
# To reload embeddings later without re-running the model:
# wp_embeddings = np.load('wp_embeddings.npy')

In [ ]:
# Run K-Means clustering with k=10
# Number of clusters can be changed in preamble

print(f"Clustering into {N_CLUSTERS} groups...")

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init='auto')
df_wp_abstracts['cluster_id'] = kmeans.fit_predict(wp_embeddings)

print("\nCluster sizes:")
print(df_wp_abstracts['cluster_id'].value_counts().sort_index())

In [ ]:
# ── INSPECT CLUSTERS TO ASSIGN LABELS ───────────────────────────────────────
# For each cluster, print:
#   - Top keywords
#   - Most common JEL codes
#   - Most common original WP categories
#   - 5 representative titles (closest to cluster centroid)

def inspect_cluster(cluster_id, df, embeddings, kmeans_model, n_titles=5):
    mask = df['cluster_id'] == cluster_id
    sub = df[mask].copy()
    sub_emb = embeddings[mask]
    
    # Titles closest to centroid
    centroid = kmeans_model.cluster_centers_[cluster_id]
    dists = np.linalg.norm(sub_emb - centroid, axis=1)
    closest_idx = np.argsort(dists)[:n_titles]
    rep_titles = sub.iloc[closest_idx]['title'].tolist()
    
    # Top JEL codes
    all_jel = ' '.join(sub['jel_codes'].tolist())
    jel_letters = [j.strip()[0] for j in all_jel.replace(',', ' ').split() if j.strip()]
    top_jel = Counter(jel_letters).most_common(5)
    
    # Top original WP categories
    all_cats = ', '.join(sub['categories'].tolist())
    cat_counts = Counter([c.strip() for c in all_cats.split(',') if c.strip()])
    top_cats = cat_counts.most_common(5)
    
    # Top keywords
    all_kw = ', '.join(sub['keywords'].tolist())
    kw_counts = Counter([k.strip().lower() for k in all_kw.split(',') if k.strip()])
    top_kw = kw_counts.most_common(8)
    
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id}  (n={len(sub):,})")
    print(f"{'='*60}")
    print(f"Top JEL letters : {top_jel}")
    print(f"Top categories: {top_cats}")
    print(f"Top keywords     : {top_kw}")
    print(f"\nRepresentative titles:")
    for i, t in enumerate(rep_titles, 1):
        print(f"  {i}. {t}")

for cid in range(N_CLUSTERS):
    inspect_cluster(cid, df_wp_abstracts, wp_embeddings, kmeans)

In [ ]:
# ── ASSIGN LABELS AFTER INSPECTING OUTPUT ABOVE ─────────────────────────────
# Edit this mapping based on what you see in the cluster inspection.
# Example labels shown below — replace with your own.

CLUSTER_LABELS = {
    0: 'Corporate/ Finance',
    1: 'Macroeconomics & Growth',
    2: 'Labor Economics',
    3: 'Public Finance',
    4: 'Trade & International',
    5: 'Public Economics',
    6: 'Microeconomics',
    7: 'Energy & Climate',
    8: 'IO',
    9: 'Education',
}

df_wp_abstracts['field_label'] = df_wp_abstracts['cluster_id'].map(CLUSTER_LABELS)

print("Label distribution:")
print(df_wp_abstracts['field_label'].value_counts())

In [ ]:
# Optional: UMAP plot to visualise the clusters (install umap-learn)
try:
    import umap
    print("Running UMAP (this takes a minute)...")
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
    coords = reducer.fit_transform(wp_embeddings)

    fig, ax = plt.subplots(figsize=(12, 8))
    palette = sns.color_palette('tab10', N_CLUSTERS)
    for cid, label in CLUSTER_LABELS.items():
        mask = df_wp_abstracts['cluster_id'] == cid
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=4, alpha=0.4, color=palette[cid], label=label)
    ax.legend(markerscale=3, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.set_title('CESifo WP Clusters (UMAP)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('cluster_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print("umap-learn not installed — skipping visualisation. Run: pip install umap-learn")

---
## STEP 3: Train a Text Classifier on Labeled Working Papers

Input: title + abstract  
Output: one of the 10 field labels  

We use **Logistic Regression on SPECTER embeddings** — fast, interpretable, and strong baseline for this kind of task.

In [ ]:
# For the classifier we use ONLY title+abstract (no JEL/keywords)
# because the published articles won't have those fields.

df_wp_abstracts['text_for_classifier'] = df_wp_abstracts['title'] + ' ' + df_wp_abstracts['abstract']

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df_wp_abstracts['field_label'])
print("Label classes:", list(le.classes_))

In [ ]:
# Re-embed using title+abstract only for the classifier
# (If your WP embeddings above already used only title+abstract, you can reuse them.)
print("Embedding title+abstract for classifier training...")

X = model.encode(
    df_wp_abstracts['text_for_classifier'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save('wp_classifier_embeddings.npy', X)
print(f"Classifier embedding shape: {X.shape}")

In [ ]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,}  |  Validation: {len(X_val):,}")

In [ ]:
# Train Logistic Regression classifier
print("Training classifier...")
clf = LogisticRegression(
    max_iter=1000,
    C=1.0,
    random_state=RANDOM_STATE,
    multi_class='multinomial'
)
clf.fit(X_train, y_train)
print("Done.")

In [ ]:
# Evaluate on validation set
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title('Validation Confusion Matrix', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save the classifier and label encoder for later reuse
import joblib
joblib.dump(clf, 'field_classifier.joblib')
joblib.dump(le,  'label_encoder.joblib')
print("Classifier saved.")